# Exploratory analysis

Read-only look at `data/warehouse.duckdb` after a batch run.

- **Working directory:** this notebook expects the Jupyter cwd to be `code/` (the directory that contains `config/pipeline.yaml`).
- **Pipeline:** run `python -m src.pipeline run-all --config config/pipeline.yaml` first (SSH / OpenSSH corpus only in v2).
- **Auth:** none. Local trusted network; do not expose the dashboard.
- **UI:** Jinja dashboard (`uvicorn src.api.main:app`), not a React app.

In [ ]:
from pathlib import Path

import pandas as pd

from src.config import load_config
from src.db.connection import get_read_only_connection

RUN_ALL_HINT = (
    "Warehouse missing or unreadable. From the code/ directory run:\n"
    "  python -m src.pipeline run-all --config config/pipeline.yaml"
)

config = load_config("config/pipeline.yaml")
conn = None
try:
    conn = get_read_only_connection(config)
    print("Connected:", Path(config["paths"]["db_path"]).resolve())
except FileNotFoundError as exc:
    print(exc)
    print(RUN_ALL_HINT)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
    print(RUN_ALL_HINT)

In [ ]:
if conn is None:
    print(RUN_ALL_HINT)
else:
    tables = conn.execute("SHOW TABLES").fetchdf()
    display(tables)
    names = {str(v).lower() for v in tables.iloc[:, 0]}
    if "incidents" not in names:
        print("No incidents table. " + RUN_ALL_HINT)
    else:
        n = conn.execute("SELECT COUNT(*) FROM incidents").fetchone()[0]
        print("incidents rows:", n)
        if n == 0:
            print("Empty incidents. Run correlate (or run-all) then refresh.")

In [ ]:
if conn is None:
    print(RUN_ALL_HINT)
else:
    try:
        incidents = conn.execute(
            """
            SELECT incident_id, entity_id, start_time, end_time,
                   severity, summary_text
            FROM incidents
            ORDER BY start_time DESC NULLS LAST, incident_id
            LIMIT 20
            """
        ).fetchdf()
    except Exception as exc:
        print(type(exc).__name__ + ":", exc)
        print(RUN_ALL_HINT)
    else:
        display(incidents)

In [ ]:
if conn is None:
    print(RUN_ALL_HINT)
else:
    entity_id = None
    try:
        row = conn.execute(
            "SELECT entity_id FROM incidents ORDER BY start_time DESC NULLS LAST LIMIT 1"
        ).fetchone()
        if row:
            entity_id = row[0]
        else:
            row = conn.execute(
                "SELECT entity_id FROM entity_bucket_counts LIMIT 1"
            ).fetchone()
            if row:
                entity_id = row[0]
    except Exception as exc:
        print(type(exc).__name__ + ":", exc)
        entity_id = None
    if not entity_id:
        print("No entity_id to plot. " + RUN_ALL_HINT)
    else:
        print("entity_id:", entity_id)
        buckets = conn.execute(
            """
            SELECT bucket_start, event_count
            FROM entity_bucket_counts
            WHERE entity_id = ?
            ORDER BY bucket_start
            """,
            [entity_id],
        ).fetchdf()
        display(buckets)

In [ ]:
metrics_path = Path("data/reports/eval/metrics.json")
if metrics_path.is_file():
    import json

    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    display(metrics)
else:
    print(
        "No data/reports/eval/metrics.json yet. After detections exist, run:\n"
        "  python -m src.pipeline evaluate --config config/pipeline.yaml"
    )